# Phase 1 — Data Cleaning & Feature Engineering
**Egypt Real Estate 2026 | PropertyFinder Dataset (39,713 listings, 53 columns)**

This notebook produces two clean, analysis-ready exports:
- `egypt_re_clean.csv` — full cleaned dataset with all engineered features
- `egypt_re_amenities_binary.csv` — binary columns for top-20 amenities

All downstream notebooks (EDA, ML, NLP, Time-Series) depend on these outputs.

## 0. Install & Import Libraries

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install(
    "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "xgboost",
    "nltk", "textblob", "vaderSentiment", "wordcloud", "langdetect",
    "scipy"
)

print("All libraries installed.")

In [ ]:
import os
import ast
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries imported successfully.")

---
## 1. Load & Inspect

We load the raw CSV (UTF-8 with BOM encoding) and immediately print shape and dtypes to understand what we are working with before touching any values.

In [ ]:
RAW_PATH = "propertyfinder.csv"   # ← adjust path if the file is in a sub-folder

df_raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig", low_memory=False)

print(f"Shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
print(df_raw.dtypes.to_string())

In [ ]:
df_raw.head(3)

### 1.1 Null Analysis — Identify Columns with >40% Missing Values

Columns above the 40% threshold are documented here. The decision for each is:
- **Drop** when the column has no plausible imputation strategy and carries little analytic value.
- **Impute** when the column is analytically important and a sensible fill (median, mode, sentinel) is available.

In [ ]:
null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ["column", "null_pct"]

high_null = null_df[null_df["null_pct"] > 40]

print(f"Columns with >40% nulls ({len(high_null)} found):")
print(high_null.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, max(4, len(null_df) * 0.3)))
colors   = ["#DC2626" if v > 40 else "#2563EB" for v in null_df["null_pct"]]
ax.barh(null_df["column"], null_df["null_pct"], color=colors)
ax.axvline(40, color="black", linestyle="--", linewidth=1, label="40% threshold")
ax.set_xlabel("Missing %")
ax.set_title("Missing Values by Column (red = >40%)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Decision log ──────────────────────────────────────────────────────────────
# Columns >40% null that we DROP (no imputation possible / not needed downstream)
# Update this list based on the actual output above.
DROP_HIGH_NULL = [
    col for col in high_null["column"]
    if col not in [
        # Keep these even if >40% null because we impute them:
        "furnished", "completion_status", "payment_method",
        "district", "subdistrict", "town"
    ]
]

print("Will DROP these high-null columns:")
for c in DROP_HIGH_NULL:
    print(f"  {c}")

---
## 2. Working Copy

We keep `df_raw` untouched throughout and work on `df`. All mutations happen on `df`.

In [ ]:
df = df_raw.copy()

# Drop high-null columns identified above
df.drop(columns=[c for c in DROP_HIGH_NULL if c in df.columns], inplace=True)

print(f"Working shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 3. Deduplication

**Strategy:** Group by `listing_id`, keep the row with the latest `scraped_at`.
This preserves the most up-to-date version of any re-scraped listing.

In [ ]:
# Parse scraped_at first so we can sort by it
if "scraped_at" in df.columns:
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)

before = len(df)

if "listing_id" in df.columns:
    df.sort_values("scraped_at", ascending=False, inplace=True, na_position="last")
    df.drop_duplicates(subset=["listing_id"], keep="first", inplace=True)
    df.reset_index(drop=True, inplace=True)

after = len(df)
print(f"Rows before dedup : {before:,}")
print(f"Rows after  dedup : {after:,}")
print(f"Duplicates removed: {before - after:,}")

---
## 4. Column-Level Cleaning

### 4.1 `offering_type` — Standardise Buy / Rent Labels

The raw field may use `category` (buy/rent lowercase) or `offering_type` (Buy/Rent capitalised). We unify both into `offering_type` with title-case values.

In [ ]:
# Some datasets store offering type in 'category', others in 'offering_type'
if "offering_type" not in df.columns and "category" in df.columns:
    df["offering_type"] = df["category"].str.strip().str.title()
elif "offering_type" in df.columns:
    df["offering_type"] = df["offering_type"].str.strip().str.title()

print(df["offering_type"].value_counts())

### 4.2 `price_egp` — IQR Outlier Removal per Offering Type

We apply the IQR fence **separately** for Buy and Rent listings because their price distributions are in completely different ranges. Rows falling outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` are dropped.

In [ ]:
df["price_egp"] = pd.to_numeric(df["price_egp"], errors="coerce")

# Remove clearly impossible prices first
df = df[df["price_egp"] > 0].copy()

def iqr_mask(series, multiplier=1.5):
    """Return boolean mask — True for rows within IQR fence."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr    = q3 - q1
    return series.between(q1 - multiplier * iqr, q3 + multiplier * iqr)

before = len(df)
masks = []
for otype in df["offering_type"].dropna().unique():
    idx  = df["offering_type"] == otype
    mask = iqr_mask(df.loc[idx, "price_egp"])
    masks.append(mask.reindex(df.index, fill_value=False))

if masks:
    combined = masks[0]
    for m in masks[1:]:
        combined = combined | m
    df = df[combined].copy()

print(f"Rows before price IQR filter : {before:,}")
print(f"Rows after  price IQR filter : {len(df):,}  (removed {before - len(df):,})")
print()
print(df.groupby("offering_type")["price_egp"].describe())

### 4.3 `bedrooms` & `bathrooms` — Parse to Numeric

Both columns are stored as object (string) in the raw CSV. We handle special values:
- `"Studio"` → 0 bedrooms
- `"10+"` → 10
- Any other non-numeric → NaN, then imputed with column median

In [ ]:
def parse_room_count(series):
    """Convert bedroom/bathroom strings to numeric."""
    s = series.astype(str).str.strip().str.lower()
    s = s.replace({"studio": "0", "10+": "10", "nan": np.nan})
    s = s.str.replace(r"[^\d.]", "", regex=True)   # strip any remaining non-digits
    s = s.replace("", np.nan)
    return pd.to_numeric(s, errors="coerce")

df["bedrooms"]  = parse_room_count(df["bedrooms"])
df["bathrooms"] = parse_room_count(df["bathrooms"])

# Fill NaN with median
df["bedrooms"]  = df["bedrooms"].fillna(df["bedrooms"].median())
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())

print("Bedrooms  — unique values:", sorted(df["bedrooms"].dropna().unique().tolist())[:15])
print("Bathrooms — unique values:", sorted(df["bathrooms"].dropna().unique().tolist())[:15])

### 4.4 `area_value` — Verify Unit & Flag Anomalies

The `area_unit` column should confirm all values are in sqm. We flag rows where `area_unit != 'sqm'` and rows with physically impossible sizes (<10 or >50,000 sqm).

In [ ]:
df["area_value"] = pd.to_numeric(df["area_value"], errors="coerce")

# Verify unit
if "area_unit" in df.columns:
    non_sqm = df["area_unit"].str.lower().ne("sqm") & df["area_unit"].notna()
    print(f"Rows with area_unit != 'sqm': {non_sqm.sum():,}")
    if non_sqm.sum() > 0:
        print(df.loc[non_sqm, "area_unit"].value_counts().head())

# Flag anomalies
df["area_flag"] = (
    (df["area_value"] < 10) | (df["area_value"] > 50_000)
).astype(int)

flagged = df["area_flag"].sum()
print(f"\nArea anomalies flagged (<10 or >50,000 sqm): {flagged:,}")
print(df.loc[df["area_flag"] == 1, "area_value"].describe())

### 4.5 `listed_date` & `scraped_at` — Parse Dates & Extract Features

Both columns are parsed to timezone-aware datetime (UTC). We then extract `year`, `month`, `week`, and `day_of_week` from `listed_date` for use in EDA and time-series notebooks.

In [ ]:
df["listed_date"] = pd.to_datetime(df["listed_date"], errors="coerce", utc=True)
# scraped_at already parsed in dedup step above

# Temporal features from listed_date
df["year"]        = df["listed_date"].dt.year
df["month"]       = df["listed_date"].dt.month
df["week"]        = df["listed_date"].dt.isocalendar().week.astype("Int64")
df["day_of_week"] = df["listed_date"].dt.dayofweek  # 0=Mon … 6=Sun

print("listed_date range:")
print(f"  Min : {df['listed_date'].min()}")
print(f"  Max : {df['listed_date'].max()}")
print(f"  Null: {df['listed_date'].isna().sum():,}")

### 4.6 `furnished` — Normalise to 3 Categories

Raw values vary (Yes/No/yes/Furnished/Semi/etc.). We map everything to exactly three values: `Furnished`, `Unfurnished`, `Semi-Furnished`. Anything unrecognised maps to `Unfurnished` (conservative default).

In [ ]:
print("Raw furnished values:")
print(df["furnished"].value_counts(dropna=False).head(20))

FURNISHED_MAP = {
    "yes":             "Furnished",
    "furnished":       "Furnished",
    "fully furnished": "Furnished",
    "no":              "Unfurnished",
    "unfurnished":     "Unfurnished",
    "semi":            "Semi-Furnished",
    "semi-furnished":  "Semi-Furnished",
    "semi furnished":  "Semi-Furnished",
    "partly":          "Semi-Furnished",
    "partially":       "Semi-Furnished",
}

df["furnished"] = (
    df["furnished"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(FURNISHED_MAP)
    .fillna("Unfurnished")   # unrecognised → Unfurnished
)

print("\nNormalised furnished values:")
print(df["furnished"].value_counts())

### 4.7 `amenities` — Parse & Create Top-20 Binary Columns

The raw `amenities` field is a string representation of a list (pipe-delimited `|` or Python list syntax). We:
1. Parse each row into an actual Python list.
2. Count the global frequency of every amenity.
3. Take the top-20 by frequency and create binary (0/1) indicator columns.

In [ ]:
def parse_amenities(val):
    """Parse a raw amenities cell into a list of stripped strings."""
    if pd.isna(val) or str(val).strip() in ("", "nan", "[]", "None"):
        return []
    s = str(val).strip()
    # Try Python list literal first
    if s.startswith("["):
        try:
            items = ast.literal_eval(s)
            return [str(i).strip() for i in items if str(i).strip()]
        except Exception:
            pass
    # Pipe-delimited fallback
    return [a.strip() for a in s.split("|") if a.strip()]

df["amenities_list"] = df["amenities"].apply(parse_amenities)

# Global amenity frequency
all_amenities = [a for lst in df["amenities_list"] for a in lst]
amenity_freq  = Counter(all_amenities)

print(f"Total unique amenities found: {len(amenity_freq)}")
print("\nTop 25 by frequency:")
for name, cnt in amenity_freq.most_common(25):
    print(f"  {cnt:>6,}  {name}")

In [ ]:
TOP_N_AMENITIES = 20
top_amenities   = [name for name, _ in amenity_freq.most_common(TOP_N_AMENITIES)]

print(f"Top-{TOP_N_AMENITIES} amenities selected:")
for i, a in enumerate(top_amenities, 1):
    print(f"  {i:2}. {a}")

def make_col_name(amenity):
    """Turn amenity name into a safe column name."""
    return "amen_" + re.sub(r"[^a-z0-9]+", "_", amenity.lower()).strip("_")

for amenity in top_amenities:
    col = make_col_name(amenity)
    df[col] = df["amenities_list"].apply(lambda lst: int(amenity in lst))

amenity_cols = [make_col_name(a) for a in top_amenities]
print(f"\nAmenity binary columns created: {amenity_cols}")

### 4.8 `description` — Strip HTML & Normalise Whitespace

Some listings include raw HTML tags in their description. We strip all tags and normalise whitespace while preserving both English and Arabic text intact.

In [ ]:
HTML_TAG_RE       = re.compile(r"<[^>]+>")
HTML_ENTITY_RE    = re.compile(r"&[a-zA-Z]{2,6};|&#?\w+;")
MULTI_WHITESPACE  = re.compile(r"[ \t]+")

def clean_description(text):
    if pd.isna(text):
        return ""
    text = HTML_TAG_RE.sub(" ", str(text))          # remove tags
    text = HTML_ENTITY_RE.sub(" ", text)             # remove HTML entities
    text = MULTI_WHITESPACE.sub(" ", text)           # collapse spaces/tabs
    text = "\n".join(line.strip() for line in text.splitlines())  # trim line edges
    return text.strip()

if "description" in df.columns:
    df["description"] = df["description"].apply(clean_description)

    # Sanity check: show a sample
    sample = df["description"].dropna().iloc[:3].tolist()
    for i, s in enumerate(sample, 1):
        print(f"--- Sample {i} (first 200 chars) ---")
        print(s[:200])
        print()
else:
    print("'description' column not found — skipping.")

---
## 5. Feature Engineering

All derived features are computed on the cleaned dataframe so they benefit from the
corrected dtypes and outlier-removed prices.

In [ ]:
# ── 5.1 price_per_sqm ─────────────────────────────────────────────────────────
# Guard against division by zero (area anomalies already flagged)
df["price_per_sqm"] = np.where(
    df["area_value"] > 0,
    df["price_egp"] / df["area_value"],
    np.nan
)

# ── 5.2 bedroom_bathroom_ratio ────────────────────────────────────────────────
df["bedroom_bathroom_ratio"] = np.where(
    df["bathrooms"] > 0,
    df["bedrooms"] / df["bathrooms"],
    np.nan
)

# ── 5.3 days_since_listed ─────────────────────────────────────────────────────
# Difference between scraped_at and listed_date in calendar days
if "scraped_at" in df.columns and "listed_date" in df.columns:
    df["days_since_listed"] = (
        df["scraped_at"] - df["listed_date"]
    ).dt.total_seconds().div(86_400).round(1)
    # Negative values (data errors) → NaN
    df.loc[df["days_since_listed"] < 0, "days_since_listed"] = np.nan

# ── 5.4 listing_age_days ──────────────────────────────────────────────────────
# Age measured from a fixed reference date (scrape date = 2026-03-05)
REF_DATE = pd.Timestamp("2026-03-05", tz="UTC")
df["listing_age_days"] = (
    REF_DATE - df["listed_date"]
).dt.total_seconds().div(86_400).round(1)
df.loc[df["listing_age_days"] < 0, "listing_age_days"] = np.nan

# ── 5.5 is_furnished_binary ───────────────────────────────────────────────────
df["is_furnished_binary"] = (df["furnished"] == "Furnished").astype(int)

# ── 5.6 amenity_count ─────────────────────────────────────────────────────────
df["amenity_count"] = df["amenities_list"].apply(len)

print("Feature engineering complete. New columns:")
new_cols = [
    "price_per_sqm", "bedroom_bathroom_ratio", "days_since_listed",
    "listing_age_days", "is_furnished_binary", "amenity_count"
]
print(df[new_cols].describe().T.round(2))

---
## 6. Final Null Check & Type Audit

In [ ]:
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)

print(f"Columns still containing nulls: {len(remaining_nulls)}")
print(remaining_nulls.to_string())

In [ ]:
# Impute remaining numerics with median, categoricals with 'Unknown'
for col in df.select_dtypes(include="number").columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if df[col].isna().any():
        df[col] = df[col].fillna("Unknown")

print(f"Nulls remaining after imputation: {df.isnull().sum().sum()}")
print(f"Final clean shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 7. Quick Distribution Check

A visual sanity check on the most important numeric columns after cleaning.

In [ ]:
check_cols = ["price_egp", "price_per_sqm", "area_value", "bedrooms", "bathrooms", "amenity_count"]
check_cols = [c for c in check_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(check_cols):
    data = df[col].dropna()
    # Use log-scale for price columns
    if "price" in col:
        data = np.log1p(data)
        xlabel = f"log1p({col})"
    else:
        xlabel = col
    axes[i].hist(data, bins=50, color="#2563EB", edgecolor="white", alpha=0.85)
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_xlabel(xlabel)
    axes[i].set_ylabel("Count")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution Check — After Cleaning", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 8. Export Outputs

### 8.1 `egypt_re_clean.csv` — Full clean dataset with all engineered columns

In [ ]:
# Drop the intermediate list column (not CSV-serialisable cleanly)
export_df = df.drop(columns=["amenities_list"], errors="ignore")

CLEAN_PATH = os.path.join(OUTPUT_DIR, "egypt_re_clean.csv")
export_df.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {CLEAN_PATH}")
print(f"Shape: {export_df.shape[0]:,} rows × {export_df.shape[1]} columns")
print(f"File size: {os.path.getsize(CLEAN_PATH) / 1_048_576:.1f} MB")

### 8.2 `egypt_re_amenities_binary.csv` — `listing_id` + top-20 amenity flags

In [ ]:
id_col = "listing_id" if "listing_id" in df.columns else df.columns[0]

amenity_export = df[[id_col] + amenity_cols].copy()

AMENITY_PATH = os.path.join(OUTPUT_DIR, "egypt_re_amenities_binary.csv")
amenity_export.to_csv(AMENITY_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {AMENITY_PATH}")
print(f"Shape: {amenity_export.shape[0]:,} rows × {amenity_export.shape[1]} columns")
print()
print("Amenity prevalence (% of listings):")
print((amenity_export[amenity_cols].mean() * 100).round(1).sort_values(ascending=False).to_string())

---
## 9. Summary

| Output | Description |
|---|---|
| `egypt_re_clean.csv` | Full cleaned dataset — feed directly into `02_eda.ipynb` |
| `egypt_re_amenities_binary.csv` | Amenity binary matrix — used by Power BI and `03_outlier_correlation.ipynb` |

**Cleaning decisions made in this notebook:**

| Step | Decision |
|---|---|
| Columns >40% null | Dropped unless imputable (see cell 1.1) |
| Duplicates | Kept latest `scraped_at` per `listing_id` |
| `price_egp` outliers | IQR 1.5× fence applied **separately** per offering type |
| `bedrooms` / `bathrooms` | Studio→0, 10+→10; NaN filled with column median |
| `area_value` anomalies | Flagged in `area_flag` column; **not dropped** (downstream notebooks decide) |
| `furnished` | Mapped to Furnished / Unfurnished / Semi-Furnished; unrecognised → Unfurnished |
| `amenities` | Parsed to list; top-20 binary columns created |
| `description` | HTML stripped; whitespace normalised; EN/AR text preserved |
| Remaining numerics | Filled with column median |
| Remaining categoricals | Filled with `'Unknown'` |